# Run CoDy

Run this notebook for the selected `DATASET` / `MODEL` combo.

This notebook:
- Runs the official `CoDy` implementation from the vendored CoDy submodule.
- Loads the matching checkpoint and explain index for the chosen combo.
- Saves official run artifacts under `resources/results/official_cody/` and finishes with the shared one-spot metrics summary.

Usage:
- Change `DATASET` and `MODEL` in the first code cell when needed.
- Run the notebook top to bottom.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

_PROJECT_CANDIDATES = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
PROJECT_ROOT = next((p for p in _PROJECT_CANDIDATES if (p / "I_explainer_benchmark").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate project root from the current working directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from I_explainer_benchmark.src.notebooks.explainer_notebook_setup import (
    initialize_explainer_notebook,
    prepare_explainer_run,
)

# Select the dataset / model combo here.
DATASET = "simulate_v1"
MODEL = "tgn"

# Shared notebook setup. Leave the rest unchanged.
BOOT = initialize_explainer_notebook("02_cody.ipynb", dataset=DATASET, model=MODEL, start=PROJECT_ROOT)
NB = BOOT.env
CFG = BOOT.config
SETTINGS = BOOT.settings
BENCHMARK_ROOT = BOOT.bench_root
REPO_ROOT = BOOT.repo_root
CODY_ROOT = BOOT.paths["CODY_ROOT"]
TEMGX_LINK = BOOT.paths["TEMGX_LINK"]


In [2]:
import torch
from I_explainer_benchmark.src.notebooks.notebook_helpers import set_global_seed

MODEL = str(MODEL).strip().lower()
SUPPORTED_MODEL_TYPES = set(CFG["supported_model_types"])
if MODEL not in SUPPORTED_MODEL_TYPES:
    raise ValueError(f"Unsupported MODEL={MODEL!r}. Choose one of {sorted(SUPPORTED_MODEL_TYPES)}")

CTX = prepare_explainer_run("02_cody.ipynb", dataset=DATASET, model=MODEL, start=PROJECT_ROOT)

DATASET_NAME = CTX.dataset
MODEL_TYPE = CTX.model
SEED = int(CFG["seed"])
set_global_seed(SEED)

SELECTION_POLICY = str(SETTINGS["selection_policy"])
CANDIDATES_SIZE = int(SETTINGS["candidates_size"])
MAX_STEPS = int(SETTINGS["max_steps"])
ALPHA = float(SETTINGS["alpha"])
APPROXIMATE_PREDICTIONS = bool(SETTINGS["approximate_predictions"])
N_EVENTS = int(SETTINGS["n_events"])
PAPER_REDDIT_TGN_EVENT_PROTOCOL = bool(SETTINGS.get("paper_reddit_tgn_event_protocol", False))
PAPER_REDDIT_LOGIT_MIN = float(SETTINGS.get("paper_reddit_logit_min", 0.2))
PAPER_REDDIT_TRAIN_START = float(SETTINGS.get("paper_reddit_train_start", 0.2))
PAPER_REDDIT_TRAIN_END = float(SETTINGS.get("paper_reddit_train_end", 0.6))

EXPLAIN_INDEX_PATH = CTX.explain_index_path
CKPT_PATH = CTX.checkpoint_path
DEVICE = CTX.device

print("DATASET:", DATASET_NAME)
print("MODEL  :", MODEL_TYPE)
print("CKPT   :", CKPT_PATH)
print("DEVICE :", DEVICE)


DATASET: simulate_v1
MODEL  : tgn
CKPT   : /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/models/simulate_v1/tgn/tgn_simulate_v1_best.pth
DEVICE : mps


In [3]:
import inspect

from cody.explainer.cody import CoDy

cody_src = (inspect.getsourcefile(CoDy) or "").replace("\\", "/")
print("CoDy source:", cody_src)
assert "/submodules/explainer/cody/cody/explainer/cody.py" in cody_src.lower(), (
    "Expected official CoDy class from submodules/explainer/cody."
)


CoDy source: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/submodules/explainer/CoDy/cody/explainer/cody.py


In [4]:
import numpy as np
import pandas as pd

from I_explainer_benchmark.src.data.io import load_processed_dataset
from I_explainer_benchmark.src.data.explain_index import load_explain_idx
from I_explainer_benchmark.src.notebooks.notebook_helpers import forced_target_event_ids_for_combo as _forced_target_event_ids_for_combo
from I_explainer_benchmark.src.models.loader import load_backbone_model

bundle = load_processed_dataset(DATASET_NAME)
events = bundle["interactions"]
edge_features = bundle.get("edge_features")
node_features = bundle.get("node_features")

if edge_features is None or node_features is None:
    raise ValueError("CoDy requires edge and node features in the processed dataset bundle.")

model, _ = load_backbone_model(
    model_type=MODEL_TYPE,
    dataset_name=DATASET_NAME,
    ckpt_path=CKPT_PATH,
    device=DEVICE,
)
backbone = model.backbone

all_event_ids = [int(v) for v in load_explain_idx(str(EXPLAIN_INDEX_PATH), start=0)]
FORCED_TARGET_EVENT_IDS = list(globals().get("FORCED_TARGET_EVENT_IDS", _forced_target_event_ids_for_combo(DATASET_NAME, MODEL_TYPE)))
target_event_ids = []
_seen_target_event_ids = set()
for _event_id in [*FORCED_TARGET_EVENT_IDS, *all_event_ids]:
    _event_id = int(_event_id)
    if _event_id in _seen_target_event_ids:
        continue
    target_event_ids.append(_event_id)
    _seen_target_event_ids.add(_event_id)
    if len(target_event_ids) >= int(N_EVENTS):
        break

print("Loaded events rows:", len(events))
print("edge_features shape:", tuple(edge_features.shape))
print("node_features shape:", tuple(node_features.shape))
print("Target event ids:", target_event_ids)


102 events to explain
Loaded events rows: 16038
edge_features shape: (16039, 4)
node_features shape: (5, 4)
Target event ids: [3, 92, 142, 268, 386, 471, 711, 935, 1056, 1161, 1266, 1457, 1526, 1750, 1864, 1985, 2174, 2430, 2590, 2889, 3081, 3357, 3466, 3537, 3674, 3876, 4025, 4167, 4291, 4465]


In [5]:
from temgxlib.data import ContinuousTimeDynamicGraphDataset
from I_explainer_benchmark.src.models.adapters.wrapper import TGNNWrapper
from I_explainer_benchmark.src.notebooks.official_explainer_notebook_runtime import prepare_cody_runtime

_cody_prepared = prepare_cody_runtime(
    events=events,
    edge_features=edge_features,
    node_features=node_features,
    dataset_name=DATASET_NAME,
    backbone=backbone,
    device=DEVICE,
    dataset_cls=ContinuousTimeDynamicGraphDataset,
    tgnn_wrapper_cls=TGNNWrapper,
)

cody_events = _cody_prepared.cody_events
edge_features_for_cody = _cody_prepared.edge_features_for_cody
model_event_ids = _cody_prepared.model_event_ids
cody_dataset = _cody_prepared.cody_dataset
num_hops = _cody_prepared.num_hops
tgnn_wrapper = _cody_prepared.tgnn_wrapper
event_id_offset = _cody_prepared.event_id_offset

print("num_hops:", num_hops)
print("event_id_offset:", event_id_offset)


num_hops: 2
event_id_offset: 1


In [6]:
from IPython.display import display
from cody.constants import COL_ID

from I_explainer_benchmark.src.notebooks.official_explainer_notebook_runtime import build_official_counterfactual_records

official_cody = CoDy(
    tgnn_wrapper=tgnn_wrapper,
    selection_policy=SELECTION_POLICY,
    candidates_size=CANDIDATES_SIZE,
    max_steps=MAX_STEPS,
    verbose=False,
    approximate_predictions=APPROXIMATE_PREDICTIONS,
    alpha=ALPHA,
)

_cody_run = build_official_counterfactual_records(
    explainer=official_cody,
    cody_events=cody_events,
    target_event_ids=target_event_ids,
    event_id_offset=event_id_offset,
    backbone=backbone,
    anchor_prefix='official_cody',
    run_id='official_cody',
    explainer_name='cody',
    candidate_id_column=COL_ID,
    row_prefix_fields={'selection_policy': SELECTION_POLICY},
    row_suffix_fields={'max_steps': int(MAX_STEPS)},
)

rows = _cody_run.rows
cody_result_records = _cody_run.result_records
results_df = _cody_run.results_df
display(results_df)


,event_idx,internal_event_id,selection_policy,candidate_size,max_steps,elapsed_sec,is_counterfactual,original_prediction,counterfactual_prediction,cf_event_ids,cf_event_importances,cf_size,candidate_event_ids
0,3,2,temporal,3,100,1.869038,False,2.700182,2.700182,[],[],0,"[1, 2, 3]"
1,92,91,temporal,64,100,3.084568,False,-3.079303,-3.079303,[],[],0,"[29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 4..."
2,142,141,temporal,64,100,2.943350,False,-5.156529,-4.797154,[131],[0.359375],1,"[79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 9..."
3,268,267,temporal,64,100,3.396577,True,2.831257,-2.408764,[264],[5.240021228790283],1,"[205, 206, 207, 208, 209, 210, 211, 212, 213, ..."
4,386,385,temporal,64,100,2.726210,False,-0.424234,-0.424234,[],[],0,"[323, 324, 325, 326, 327, 328, 329, 330, 331, ..."
5,471,470,temporal,64,100,2.770546,False,-4.041662,-4.041662,[],[],0,"[408, 409, 410, 411, 412, 413, 414, 415, 416, ..."
6,711,710,temporal,64,100,2.667249,False,-1.557502,-1.557502,[],[],0,"[648, 649, 650, 651, 652, 653, 654, 655, 656, ..."
7,935,934,temporal,64,100,2.700124,False,-3.318067,-3.318067,[],[],0,"[872, 873, 874, 875, 876, 877, 878, 879, 880, ..."
8,1056,1055,temporal,64,100,2.724010,False,-3.466963,-3.466963,[],[],0,"[993, 994, 995, 996, 997, 998, 999, 1000, 1001..."
9,1161,1160,temporal,64,100,2.776944,False,-5.023037,-4.896242,[1141],[0.12679529190063477],1,"[1098, 1099, 1100, 1101, 1102, 1103, 1104, 110..."


In [7]:
# Shared metrics + export pipeline (clean notebook wrapper)
from I_explainer_benchmark.src.notebooks.notebook_metrics_pipeline import run_notebook_metrics_from_namespace

_pipeline_out = run_notebook_metrics_from_namespace(globals(), CFG)
globals().update(_pipeline_out)
run_dir_export = _pipeline_out['run_dir_export']
export_root = _pipeline_out['export_root']
out_jsonl = _pipeline_out['out_jsonl']
out_dir = _pipeline_out['out_dir']
base_name = _pipeline_out['base_name']
print('CURRENT_RUN_ID:', _pipeline_out['CURRENT_RUN_ID'])
print('Saved run export directory:', run_dir_export)
print('Updated global tables under:', export_root)


CURRENT_RUN_ID: simulate_v1_tgn_official_cody_20260315_200849
Saved run export directory: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/summary_ready/simulate_v1_tgn_official_cody_20260315_200849
Updated global tables under: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/summary_ready


In [8]:
# One-spot metrics summary (official)
from I_explainer_benchmark.src.notebooks.notebook_display import show_one_spot_metrics_summary

_one_spot = globals().get("one_spot", globals().get("_pipeline_out", {}).get("one_spot", {}))
show_one_spot_metrics_summary(_one_spot)


One-spot metrics summary (official):


,dataset,model,explainer,variant,n_events,best_fid,best_fid_sparsity,aufsc,best_minus_aufsc,best_fid_raw,...,tempme_acc_auc.ratio_acc,tempme_acc_auc.ratio_prob,tempme_acc_auc.ratio_logit,tempme_acc_auc.ratio_aps,tempme_acc_auc.ratio_auc,temgx_aufsc,cody_AUFSC_plus,cody_AUFSC_minus,cody_CHARR,elapsed_sec
0,simulate_v1,tgn,cody,official,30,1.2045290634,1.0,-0.5816672746,1.786196338,1.2045290634,...,0.778125,-0.0107126486,-0.136326218,0.8322916667,0.9375,0.0933825613,0.7382147182,0.3839309326,0.5051455932,2.761644582
